# 06A v2 — Alternative Model Benchmark Finalization

This notebook performs **no model training**. It restores the five completed pre-EBM runtime tables from the protected backup, adds the five EBM fold timings, regenerates the corrected compute–performance figure, and writes a publication-ready v2 report. All prediction metrics and bootstrap results are read from the completed 06A benchmark.


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "06A_V2_ALTERNATIVE_MODEL_BENCHMARK_FINALIZE"

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "alternative_model_benchmark_v1"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
BACKUP_RUNTIME_PATH = OUTPUT_ROOT / "ALTERNATIVE_MODEL_RUNTIME_PRE_EBM_BACKUP_V1.csv"
CURRENT_RUNTIME_PATH = OUTPUT_ROOT / "ALTERNATIVE_MODEL_RUNTIME_V1.csv"
METRICS_PATH = OUTPUT_ROOT / "ALTERNATIVE_MODEL_OOF_METRICS_V1.csv"
BOOTSTRAP_PATH = OUTPUT_ROOT / "PAIRED_INCIDENT_BOOTSTRAP_VS_LIGHTGBM_V1.csv"
FOLD_METRICS_PATH = OUTPUT_ROOT / "ALTERNATIVE_MODEL_OUTER_FOLD_METRICS_V1.csv"
SPECS_PATH = OUTPUT_ROOT / "ALTERNATIVE_MODEL_FIXED_SPECS_V1.json"
V1_CHECKS_PATH = OUTPUT_ROOT / "FINAL_ALTERNATIVE_MODEL_BENCHMARK_CHECKS_V1.csv"

for path in [BACKUP_RUNTIME_PATH, CURRENT_RUNTIME_PATH, METRICS_PATH, BOOTSTRAP_PATH, FOLD_METRICS_PATH, SPECS_PATH, V1_CHECKS_PATH]:
    assert path.exists(), path

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 240, "axes.spines.top": False, "axes.spines.right": False})
print("RUNNING:", NOTEBOOK_BUILD_ID)


## 1. Restore the complete checkpointed runtime ledger


In [ ]:
backup_runtime = pd.read_csv(BACKUP_RUNTIME_PATH)
current_runtime = pd.read_csv(CURRENT_RUNTIME_PATH)

pre_ebm_models = [
    "XGBOOST", "CATBOOST", "HIST_GRADIENT_BOOSTING",
    "ELASTIC_NET_LOGISTIC", "EXTRA_TREES",
]
pre_ebm = backup_runtime[backup_runtime["model_id"].isin(pre_ebm_models)].copy()
ebm = current_runtime[current_runtime["model_id"].eq("EBM")].copy()
runtime_v2 = pd.concat([pre_ebm, ebm], ignore_index=True)
runtime_v2["outer_fold"] = runtime_v2["outer_fold"].astype(int)
runtime_v2 = runtime_v2.sort_values(["model_id", "outer_fold"]).reset_index(drop=True)

assert len(pre_ebm) == 25
assert len(ebm) == 5
assert len(runtime_v2) == 30
assert not runtime_v2.duplicated(["model_id", "outer_fold"]).any()
assert runtime_v2.groupby("model_id")["outer_fold"].nunique().eq(5).all()
for column in ["preprocessing_seconds", "fit_seconds", "predict_seconds"]:
    assert np.isfinite(runtime_v2[column]).all(), column

runtime_v2["status"] = "trained_checkpointed"
runtime_v2_path = OUTPUT_ROOT / "ALTERNATIVE_MODEL_RUNTIME_V2.csv"
runtime_v2.to_csv(runtime_v2_path, index=False)

runtime_summary = runtime_v2.groupby("model_id", as_index=False).agg(
    preprocessing_minutes=("preprocessing_seconds", lambda x: float(x.sum() / 60)),
    fit_minutes=("fit_seconds", lambda x: float(x.sum() / 60)),
    prediction_minutes=("predict_seconds", lambda x: float(x.sum() / 60)),
).sort_values("fit_minutes")
runtime_summary.to_csv(OUTPUT_ROOT / "ALTERNATIVE_MODEL_RUNTIME_SUMMARY_V2.csv", index=False)
display(runtime_v2)
display(runtime_summary)


## 2. Regenerate the corrected compute–performance figure


In [ ]:
model_metrics = pd.read_csv(METRICS_PATH)
runtime_plot = runtime_summary.merge(
    model_metrics[["model_id", "incident_macro_brier", "mixed_incident_macro_roc_auc"]],
    on="model_id", how="left", validate="one_to_one",
)
assert len(runtime_plot) == 6 and runtime_plot["incident_macro_brier"].notna().all()

MODEL_COLORS = {
    "XGBOOST": "#dd8452", "CATBOOST": "#55a868",
    "HIST_GRADIENT_BOOSTING": "#c44e52", "ELASTIC_NET_LOGISTIC": "#8172b3",
    "EXTRA_TREES": "#937860", "EBM": "#64b5cd",
}

fig, ax = plt.subplots(figsize=(9, 6.5), constrained_layout=True)
for _, row in runtime_plot.iterrows():
    ax.scatter(
        row["fit_minutes"], row["incident_macro_brier"],
        s=85, color=MODEL_COLORS[row["model_id"]], label=row["model_id"],
    )
ax.set_xscale("log")
ax.set_xlabel("Total five-fold fit time (minutes; log scale)")
ax.set_ylabel("Incident-macro Brier score (lower is better)")
ax.set_title("Compute–performance tradeoff across alternative models")
ax.grid(alpha=0.2)
ax.legend(title="Model", frameon=False, fontsize=8, title_fontsize=9, ncol=2, loc="upper right")
fig.savefig(FIGURE_ROOT / "FIG_06A_03_RUNTIME_VS_PERFORMANCE_V2.png", bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "FIG_06A_03_RUNTIME_VS_PERFORMANCE_V2.pdf", bbox_inches="tight")
plt.show()


## 3. Write the corrected final checks and report


In [ ]:
paired_bootstrap = pd.read_csv(BOOTSTRAP_PATH)
fold_metrics = pd.read_csv(FOLD_METRICS_PATH)
v1_checks = pd.read_csv(V1_CHECKS_PATH)
with SPECS_PATH.open("r", encoding="utf-8") as f:
    model_specs = json.load(f)

checks_v2 = pd.concat([
    v1_checks,
    pd.DataFrame([
        {"check": "exact_30_model_fold_runtime_rows", "pass": len(runtime_v2) == 30},
        {"check": "all_runtime_values_restored_and_finite", "pass": bool(np.isfinite(runtime_v2[["preprocessing_seconds", "fit_seconds", "predict_seconds"]]).all().all())},
        {"check": "corrected_runtime_figure_png_exists", "pass": (FIGURE_ROOT / "FIG_06A_03_RUNTIME_VS_PERFORMANCE_V2.png").exists()},
        {"check": "corrected_runtime_figure_pdf_exists", "pass": (FIGURE_ROOT / "FIG_06A_03_RUNTIME_VS_PERFORMANCE_V2.pdf").exists()},
    ]),
], ignore_index=True)
PASS = bool(checks_v2["pass"].astype(bool).all())
checks_v2.to_csv(OUTPUT_ROOT / "FINAL_ALTERNATIVE_MODEL_BENCHMARK_CHECKS_V2.csv", index=False)

def md_table(frame):
    try:
        return frame.to_markdown(index=False)
    except Exception:
        return frame.to_string(index=False)

report_lines = [
    "# 06A Alternative Tabular Model Benchmark — Finalized v2",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    f"Generated UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "## Scientific status",
    "",
    "Post-freeze model-family robustness experiment. The frozen 05C LightGBM paper primary was not changed. All alternatives used the same 803-feature contract, ordinary row weighting, and five incident-disjoint outer folds. Hyperparameters were prespecified, with no recalibration or threshold selection.",
    "",
    "## OOF model comparison",
    "",
    md_table(model_metrics),
    "",
    "## Paired incident bootstrap versus frozen LightGBM",
    "",
    md_table(paired_bootstrap),
    "",
    "## Complete runtime summary",
    "",
    md_table(runtime_summary),
    "",
    "## Final checks",
    "",
    md_table(checks_v2),
    "",
    "## Decision",
    "",
    "`PASS / ALTERNATIVE MODEL BENCHMARK COMPLETE; PAPER PRIMARY REMAINS FROZEN`" if PASS else "`REVIEW REQUIRED`",
]
report_path = OUTPUT_ROOT / "ALTERNATIVE_MODEL_BENCHMARK_REPORT_V2.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
display(checks_v2)
print()
print("06A V2 FINALIZATION:", "PASS" if PASS else "REVIEW REQUIRED")
print("Saved:", report_path)
print("DONE:", NOTEBOOK_BUILD_ID)
